# Задача: Определение занятости парковочного места
## Обучение и валидация моделей классификации изображений

### Импорт библиотек

In [1]:
from model import create_model, train_model

import json
from collections import Counter
from pathlib import Path

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


### Проверка сбалансированнности разделения выборки на тренировочную, валидационную и тестовую
### Проверка сбалансированности классов в каждой выборке

In [2]:
with open("dataset/annotations.json") as f:
    ann = json.load(f)

for split in ann:
    labels = ann[split]["occupancy_list"]

    print(f"\n{split}")
    print(f"Всего: {len(labels)}")
    print(Counter(labels))


train
Всего: 4081
Counter({True: 2338, False: 1743})

valid
Всего: 687
Counter({False: 369, True: 318})

test
Всего: 533
Counter({False: 331, True: 202})


### Создание класса датасета для парковочных мест

In [3]:
class ParkingDataset(Dataset):
    def __init__(
        self,
        image_dir,
        file_names,
        labels,
        transform=None
    ):
        self.image_dir = Path(image_dir)
        self.file_names = file_names
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        image_path = self.image_dir / self.file_names[idx]

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        label = int(self.labels[idx])

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, torch.tensor(label, dtype=torch.long)

### Инициализация аугментации на этапе обучения и валидации

In [4]:

train_transform = A.Compose([
    
    A.RandomResizedCrop(
        size=(224, 224),
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1),
        p=1.0
    ),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(p=0.6),
    A.HueSaturationValue(p=0.3),
    A.RandomGamma(p=0.3),

    A.GaussNoise(p=0.3),
    A.ImageCompression(quality_range=(60, 100), p=0.3),

    A.MotionBlur(p=0.2),

    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),

    ToTensorV2()
])

valid_transform = A.Compose([
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
        p=1.0
    ),

    ToTensorV2()
])

### Инициализация датасета и даталоадера для обучения и валидации

In [ ]:
train_files = ann["train"]["file_names"]
train_labels = ann["train"]["occupancy_list"]

valid_files = ann["valid"]["file_names"]
valid_labels = ann["valid"]["occupancy_list"]

In [ ]:
train_dataset = ParkingDataset(
    image_dir="dataset/images",
    file_names=train_files,
    labels=train_labels,
    transform=train_transform
)

valid_dataset = ParkingDataset(
    image_dir="dataset/images",
    file_names=valid_files,
    labels=valid_labels,
    transform=valid_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
)

### Обучение и валидация моделей классификации изображений парковки

In [ ]:
model_names = [
    "resnet18",
    "efficientnet_v2_m",
    "convnext_base",
    "vit_b_32",
    "densenet169",
]

for name in model_names:
    print(f"\n===== Training {name} =====")

    model = create_model(name)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    train_model(
        model,
        train_loader,
        valid_loader,
        optimizer,
        criterion,
        epochs=10
    )


===== Training resnet18 =====
Epoch 1: loss=28.815, F1=0.965
Epoch 2: loss=18.238, F1=0.958
Epoch 3: loss=15.318, F1=0.974
Epoch 4: loss=14.937, F1=0.983
Epoch 5: loss=11.584, F1=0.979
Epoch 6: loss=11.149, F1=0.966
Epoch 7: loss=11.141, F1=0.972
Epoch 8: loss=11.424, F1=0.978
Epoch 9: loss=9.010, F1=0.986
Epoch 10: loss=9.294, F1=0.971

===== Training efficientnet_v2_m =====
Epoch 1: loss=24.904, F1=0.978
Epoch 2: loss=11.814, F1=0.984
Epoch 3: loss=11.277, F1=0.986
Epoch 4: loss=8.533, F1=0.978
Epoch 5: loss=9.011, F1=0.981
Epoch 6: loss=7.052, F1=0.981
Epoch 7: loss=7.088, F1=0.988
Epoch 8: loss=7.393, F1=0.987
Epoch 9: loss=7.670, F1=0.991
Epoch 10: loss=5.041, F1=0.986

===== Training convnext_base =====
Epoch 1: loss=24.357, F1=0.974
Epoch 2: loss=13.093, F1=0.984
Epoch 3: loss=11.385, F1=0.973
Epoch 4: loss=10.347, F1=0.978
Epoch 5: loss=8.842, F1=0.984
Epoch 6: loss=8.560, F1=0.981
Epoch 7: loss=8.367, F1=0.984
Epoch 8: loss=8.718, F1=0.988
Epoch 9: loss=6.387, F1=0.981
Epoch 